# 00 — Setup & Data Ingest

**Purpose:** Create the Unity Catalog catalog/schemas for this project and land the raw
*Medical Cost Personal Datasets* (Kaggle) insurance table as a clean Delta table.

### Dataset
- **Kaggle:** [Medical Cost Personal Datasets](https://www.kaggle.com/datasets/mirichoi0218/insurance)
  by `mirichoi0218` — 1,338 rows, 7 columns (`age, sex, bmi, children, smoker, region, charges`).
  Small enough to run instantly on Databricks Free Edition serverless compute.

### What This Notebook Does
1. Creates the `insurance_rag_agent` catalog and its `raw` / `docs` / `agent_tools` schemas.
2. Creates a Unity Catalog Volume and reads the uploaded `insurance.csv` from it.
3. Writes `insurance_rag_agent.raw.policies_raw` with ingestion lineage columns.

### Source & Target
| Direction | Table / Path |
|-----------|--------------|
| Source | `/Volumes/insurance_rag_agent/raw/source_data/insurance.csv` |
| Target | `insurance_rag_agent.raw.policies_raw` |

> **Prerequisites:** Download `insurance.csv` from Kaggle and upload it to the Volume path
> above via **Catalog Explorer > Volumes > Upload** before running this notebook.

---

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG            = 'insurance_rag_agent'
RAW_SCHEMA          = 'raw'
DOCS_SCHEMA         = 'docs'
AGENT_TOOLS_SCHEMA  = 'agent_tools'
VOLUME_NAME         = 'source_data'

VOLUME_PATH  = f'/Volumes/{CATALOG}/{RAW_SCHEMA}/{VOLUME_NAME}'
SOURCE_FILE  = f'{VOLUME_PATH}/insurance.csv'
RAW_TABLE    = f'{CATALOG}.{RAW_SCHEMA}.policies_raw'

spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOG}')
for schema in [RAW_SCHEMA, DOCS_SCHEMA, AGENT_TOOLS_SCHEMA]:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}.{VOLUME_NAME}')

# Idempotency — drop the raw table before re-running
spark.sql(f'DROP TABLE IF EXISTS {RAW_TABLE}')

print(f'Catalog: {CATALOG}')
print(f'Source file: {SOURCE_FILE}')
print(f'Target: {RAW_TABLE}')

In [0]:
from pyspark.sql import functions as F

---

In [0]:
df = (
    spark.read
    .option('header',      'true')
    .option('inferSchema', 'true')
    .csv(SOURCE_FILE)
)

display(df.limit(5))
df.printSchema()

---

In [0]:
out_df = (
    df
    .withColumn('ingestion_timestamp', F.current_timestamp())
    .withColumn('source_file',         F.lit(SOURCE_FILE))
)

---

In [0]:
(out_df.write.format('delta').mode('overwrite').saveAsTable(RAW_TABLE))

print(f'Rows written: {out_df.count()}')

---

In [0]:
%sql
SELECT * FROM insurance_rag_agent.raw.policies_raw LIMIT 10

In [0]:
print(f'Total rows in {RAW_TABLE}: {spark.read.table(RAW_TABLE).count()}')